# ProboSed — 03: Parameter Sensitivity Analysis

**IODP Expedition 405 — Site C0019J, Japan Trench**

Tests whether the qualitative conclusion — that fault coupling substantially
increases slope failure probability — is robust across physically plausible
parameter ranges.

One parameter is varied at a time while all others are held at base values.

**Physical parameter ranges are informed by:**
- Exp 405: Vp, resistivity at C0019 (sigma_q range)
- Exp 343: overconsolidation, permeability (gamma range)
- Fulton et al. 2013: ~50 m coseismic slip (slip_mag range)
- Exp 386: seismic strengthening (gamma high end)

In [ ]:
# Cell 1: Install dependencies
!pip install numpy matplotlib pandas --quiet

In [ ]:
# Cell 2: Mount Drive and set up ProboSed
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
repo_url  = 'https://github.com/yourusername/ProboSed.git'  # update this
repo_path = '/content/ProboSed'
result = subprocess.run(['git', 'clone', repo_url, repo_path],
                        capture_output=True, text=True)
if result.returncode != 0:
    subprocess.run(['git', '-C', repo_path, 'pull'], capture_output=True)
sys.path.insert(0, repo_path)
print('ProboSed modules available')

In [ ]:
# Cell 3: Configure output
import os
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Cell 4: Import and define base parameters
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from slope.stability import run_ensemble, run_sensitivity

BASE_PARAMS = dict(
    T=10.0, dt=0.01,
    gamma=1.0, sigma_q=0.6, sigma_s=0.5,
    k_f=1.0, alpha=0.5, threshold=1.0,
    slip_mag=3.0, mainshock_t=5.0, seed=42
)

# parameter grid — physically motivated ranges
# see stability.py run_sensitivity() docstring for full justification
PARAM_GRID = {
    'alpha':     [0.2, 0.3, 0.5, 0.7, 0.9],   # coupling strength
    'sigma_q':   [0.3, 0.4, 0.6, 0.8, 1.0],   # slope noise amplitude
    'slip_mag':  [1.0, 2.0, 3.0, 4.0, 5.0],   # mainshock impulse
    'gamma':     [0.5, 0.75, 1.0, 1.5, 2.0],  # slope damping
    'threshold': [0.33, 0.67, 1.0, 1.33, 2.0], # VCD scores 0-3 mapped
}
print('Parameters defined')

In [ ]:
# Cell 5: Run sensitivity analysis
# Each parameter is varied across its grid while all others stay at base values.
# N_paths=3000 balances statistical stability with compute time in Colab.
print('Running sensitivity analysis...')
df = run_sensitivity(PARAM_GRID, BASE_PARAMS, N_paths=3000)

out_csv = os.path.join(OUTPUT_DIR, 'sensitivity_results.csv')
df.to_csv(out_csv, index=False)
print(f'\nResults saved: {out_csv}')
df

In [ ]:
# Cell 6: Compute baseline P(failure) for reference
_, _, base_p, _ = run_ensemble(N_paths=3000, **BASE_PARAMS)
print(f'Baseline P(failure) [fault-coupled, base params]: {base_p:.3f}')

## Sensitivity Figure

Each panel shows P(failure) vs one parameter value.
The red dashed line marks the baseline failure probability.
The gray dashed line marks the base parameter value.

If all panels show P(failure) above the slope-only baseline (~0.38)
across most of the parameter range, the qualitative conclusion is robust.

In [ ]:
# Cell 7: Sensitivity figure
params       = list(PARAM_GRID.keys())
n_params     = len(params)
param_labels = {
    'alpha':     'alpha (coupling)',
    'sigma_q':   'sigma_q (slope noise)',
    'slip_mag':  'Slip magnitude',
    'gamma':     'gamma (damping)',
    'threshold': 'theta (failure threshold)',
}

fig, axes = plt.subplots(1, n_params, figsize=(4 * n_params, 5))
fig.patch.set_facecolor('white')

slope_only_baseline = 0.38   # from slope-only run in notebook 02

for ax, param in zip(axes, params):
    ax.set_facecolor('white')
    subset = df[df['parameter'] == param].sort_values('value')
    ax.plot(subset['value'], subset['p_fail'], 'o-', color='navy', lw=2, ms=8, zorder=3)
    ax.axhline(base_p, color='red', ls='--', lw=1.5, alpha=0.7,
               label=f'Baseline = {base_p:.2f}')
    ax.axhline(slope_only_baseline, color='gray', ls=':', lw=1.5, alpha=0.7,
               label=f'Slope-only = {slope_only_baseline:.2f}')
    ax.axvline(BASE_PARAMS[param], color='steelblue', ls='--', lw=1.5, alpha=0.7,
               label=f'Base = {BASE_PARAMS[param]}')
    for _, row in subset.iterrows():
        ax.annotate(f"{row['p_fail']:.2f}",
                    xy=(row['value'], row['p_fail']),
                    xytext=(0, 8), textcoords='offset points',
                    ha='center', fontsize=8, color='navy')
    ax.set_xlabel(param_labels.get(param, param), fontsize=11)
    ax.set_ylabel('P(failure)' if param == params[0] else '', fontsize=11)
    ax.set_title(param_labels.get(param, param), fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    'Parameter Sensitivity Analysis — Japan Trench Slope Stability Model\n'
    'One parameter varied at a time; all others held at base values\n'
    'Site C0019J (IODP Expeditions 343, 386, 405)',
    fontsize=11, fontweight='bold', y=1.02
)
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, 'sensitivity_analysis.png')
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Cell 8: Robustness summary
# What fraction of tested values produce P(failure) above the slope-only baseline?
print('Robustness check: fraction of parameter values where P(failure) > slope-only baseline')
print(f'(Slope-only baseline = {slope_only_baseline})')
print()
for param in params:
    subset  = df[df['parameter'] == param]
    robust  = subset[subset['p_fail'] > slope_only_baseline]
    pct     = 100 * len(robust) / len(subset)
    print(f'  {param:<15}: {len(robust)}/{len(subset)} values ({pct:.0f}%)')